# ML Regression — Complete Study Reference

---

## 📐 Linear Regression

### 1. What assumptions does linear regression make?

| Assumption | What it means |
|---|---|
| **Linearity** | The relationship between features and target is linear |
| **Independence** | Observations are independent of each other |
| **Homoscedasticity** | Residuals have constant variance across all values |
| **Normality of residuals** | Errors are normally distributed (matters for inference) |
| **No multicollinearity** | Features are not highly correlated with each other |

> Violating these doesn't always break prediction — but it breaks statistical guarantees (confidence intervals, p-values).

---

### 2. Why can multicollinearity hurt linear regression?

When two features are highly correlated, the model can't figure out **which one is responsible** for the target change. Coefficients become **unstable** — tiny changes in data cause huge swings in weights. The model still fits training data okay, but coefficients lose meaning and generalization suffers.

---

### 3. MAE vs MSE vs RMSE vs R²

| Metric | Formula | Key Property | When to use |
|---|---|---|---|
| **MAE** | mean(\|y − ŷ\|) | Linear penalty — treats all errors equally | Robust to outliers |
| **MSE** | mean((y − ŷ)²) | Squares errors — big mistakes penalized hard | When large errors are very bad |
| **RMSE** | √MSE | Same units as target, still penalizes large errors | Most interpretable MSE variant |
| **R²** | 1 − SS_res/SS_tot | Proportion of variance explained (0–1) | Comparing models on same dataset |

> R² = 1 → perfect fit. R² = 0 → model is no better than just predicting the mean. R² can be **negative** if the model is worse than that baseline.

---

### 4. Why does linear regression get affected by outliers?

OLS (Ordinary Least Squares) minimizes **squared** errors. An outlier creates a massive squared residual, so the model bends its line to reduce that one point's error — at the cost of fitting the rest of the data worse. MAE-based regression is more robust because errors aren't squared.

---

### 5. Why do we scale features before regularized regression?

Regularization penalizes **coefficient size**. If one feature is in thousands (e.g., salary) and another is in decimals (e.g., rate), their coefficients are on different scales — the penalty hits them unequally. Scaling puts all features on the same playing field so the regularizer treats them fairly.

---

### 6. When would linear regression outperform complex models?

- Small dataset (complex models overfit, linear generalizes)
- Truly linear relationship in the data
- Need full interpretability (audits, regulations, explainability)
- Low signal-to-noise ratio (simple models don't learn noise)
- Baseline benchmarking

> **Rule of thumb**: If a complex model doesn't significantly beat linear regression, ship the linear model.

---

### 7. Why can high train R² but low test R² happen?

**Overfitting.** The model memorized the training data — including its noise — rather than learning the true signal. On unseen data the noise pattern doesn't repeat, so predictions fail. Signs: very high train R², much lower test R², large number of features, small training set.

---

### 8. What does a coefficient mean?

A coefficient βᵢ means: **holding all other features constant**, a one-unit increase in feature *i* is associated with a βᵢ change in the target.

Example: `price = 50,000 + 200×sqft − 3000×age`  
→ Each extra sq ft adds $200 to price, holding age constant.

> Coefficients are only interpretable when features are independent. Multicollinearity corrupts this.

---

### 9. Why is interpretability considered a strength of linear regression?

Each prediction is a **weighted sum** — you can directly read off which features matter and by how much. You can explain any single prediction: "price went up because sqft is high (+$60k) and crime rate is low (+$10k)." Regulators, doctors, and clients often require this kind of explanation.

---

### 10. What happens if features are highly correlated?

- Coefficients become **unreliable** (high variance) — they can even flip signs
- Standard errors inflate → statistical tests become meaningless
- Model predictions can still be okay, but **individual coefficients cannot be trusted**
- Solution: remove one of the correlated features, use PCA, or switch to **Ridge regression**

---
---

## 🏔️ Ridge Regression

### 1. Why was Ridge created?

To solve the **multicollinearity problem** in linear regression. When features are correlated, OLS coefficients explode in variance. Ridge adds an L2 penalty to the loss function, forcing coefficients to stay small and stable:

```
Loss = MSE + α × Σ(βᵢ²)
```

---

### 2. What does alpha control?

Alpha (λ) controls the **strength of regularization** — the trade-off between fitting the data and keeping coefficients small.

```
Low α  → closer to plain linear regression (less regularization)
High α → coefficients shrink harder toward zero (more regularization)
```

---

### 3. What happens when alpha is too small or too large?

| Alpha | Effect |
|---|---|
| **Too small** | Almost no regularization → can still overfit, multicollinearity still hurts |
| **Too large** | Coefficients are crushed toward zero → model underfits, high bias |
| **Just right** | Bias-variance sweet spot → best generalization |

> Tune alpha with cross-validation.

---

### 4. Why does Ridge help with multicollinearity?

Ridge doesn't pick between correlated features — it **shares the coefficient weight evenly** across them. Instead of one feature getting a huge coefficient and another near-zero (which is unstable), both get moderate, stable coefficients. This reduces variance without eliminating features.

---

### 5. LinearRegression vs Ridge

| | LinearRegression | Ridge |
|---|---|---|
| **Penalty** | None | L2 (sum of squared coefficients) |
| **Coefficients** | Can be large/unstable | Shrunk toward zero |
| **Multicollinearity** | Breaks down | Handles gracefully |
| **Feature selection** | No | No (just shrinks) |
| **When to use** | Clean, independent features | Real-world noisy/correlated data |

---

### 6. Does Ridge remove features completely?

**No.** Ridge shrinks coefficients toward zero but never *to* zero. All features remain in the model. If you need feature elimination, use **Lasso**.

---

### 7. Why is Ridge often better than plain linear regression on real-world noisy data?

Real data is almost always noisy and features are rarely perfectly independent. Ridge's regularization prevents the model from **overfitting to noise**. The slight bias Ridge introduces is almost always worth the large reduction in variance. The result is better generalization on test data.

---

### 8. Why should scaling happen before Ridge?

Ridge penalizes the **magnitude** of coefficients. Without scaling, a feature measured in thousands has a naturally small coefficient, and one in decimals has a large one — so Ridge punishes the wrong features. Scaling ensures the penalty is applied equally and fairly across all features.

---

### 9. When would Ridge be preferred over Lasso?

- When **all features are expected to matter** (no need for sparsity)
- When features are **highly correlated** (Lasso picks one arbitrarily; Ridge distributes)
- When you want **stable, smooth coefficient estimates**
- When you can't afford to accidentally zero out an important feature

---

### 10. How does Ridge reduce overfitting?

By penalizing large coefficients, Ridge prevents the model from fitting idiosyncratic noise in training data. A model that overfits assigns huge weights to specific features to perfectly match training samples — Ridge makes that expensive. The model is forced to stay general.

---
---

## 🔦 Lasso Regression

### 1. Main difference between Ridge and Lasso?

| | Ridge | Lasso |
|---|---|---|
| **Penalty type** | L2: Σ(βᵢ²) | L1: Σ(\|βᵢ\|) |
| **Coefficient behavior** | Shrinks toward zero | Can become **exactly zero** |
| **Feature selection** | No | **Yes** |
| **Best for** | All features relevant | Some features irrelevant |

---

### 2. Why does Lasso create sparsity?

The L1 penalty creates a loss surface with **sharp corners at zero**. The optimization solution tends to land exactly on those corners — meaning some coefficients become exactly 0. L2 (Ridge) has a smooth circular surface; it approaches zero asymptotically but never touches it.

Geometrically: L1 constraint is a diamond; L2 is a sphere. The solution often hits the **corners of the diamond** (where a coefficient = 0).

---

### 3. What does "feature selection" mean?

Feature selection means **automatically identifying which features matter** and which don't. Lasso does this implicitly: it zeros out irrelevant or redundant features. You end up with a simpler model that uses only the most predictive features — without manually testing each one.

---

### 4. Why can Lasso become unstable with highly correlated features?

When two features carry the same information, Lasso **arbitrarily picks one** and zeros the other. Which one it picks can change dramatically with small data changes. This is unpredictable and unstable. Ridge would have split the weight between both more consistently. ElasticNet solves this.

---

### 5. When is Lasso useful?

- You have **many features** and suspect only a few matter
- You want an **interpretable, sparse** model
- You want built-in **feature selection** without a separate step
- High-dimensional data (more features than samples)

---

### 6. Why might Lasso hurt performance sometimes?

- If it zeros out a feature that actually **matters a little** → underfitting
- With correlated features: the surviving feature might be noisier than the one eliminated
- Too high alpha → too many zeroed features → model becomes too simple

---

### 7. Sparse model vs dense model

| | Sparse Model | Dense Model |
|---|---|---|
| **Definition** | Most coefficients = 0; few features used | All features have non-zero coefficients |
| **Interpretability** | High | Lower |
| **Risk** | May miss subtle signals | Can overfit on noise |
| **Example** | Lasso | Ridge / Linear Regression |

---

### 8. What happens if alpha becomes too large in Lasso?

More and more coefficients get zeroed out. Eventually, **all** coefficients become zero — the model predicts the mean for every input. Complete underfitting. Test and train performance both collapse.

---
---

## ⚡ ElasticNet

### 1. Why was ElasticNet created?

To get **the best of both Ridge and Lasso**. Lasso is unstable with correlated features (picks arbitrarily) and can zero out too many. Ridge doesn't do feature selection. ElasticNet combines both penalties:

```
Loss = MSE + α × [l1_ratio × Σ|βᵢ| + (1 − l1_ratio) × Σβᵢ²]
```

---

### 2. When is ElasticNet preferred over pure Lasso?

- When features are **correlated** — ElasticNet handles groups of correlated features rather than arbitrarily eliminating them
- When you want **some** sparsity (feature selection) but not as aggressive as Lasso
- When Lasso produces unstable, randomly-varying feature selections across CV folds
- High-dimensional data with feature groups

---

### 3. What does `l1_ratio` control?

`l1_ratio` sets the **mix** between L1 (Lasso) and L2 (Ridge) penalties:

| l1_ratio | Behavior |
|---|---|
| `0` | Pure Ridge (no sparsity) |
| `1` | Pure Lasso (full sparsity, can be unstable) |
| `0.5` | Equal mix: sparse but stable |

> Treat `l1_ratio` as another hyperparameter to tune with cross-validation.

---
---

## 🧰 General ML Questions

### 1. Why use pipelines?

Pipelines **chain preprocessing and modeling** into one object. Benefits:
- Prevents **data leakage** (fit on train only)
- Reproducibility — same steps always in the same order
- Clean cross-validation — scaler fits only on each fold's training data
- Simpler deployment — one object to save/load

---

### 2. Why use ColumnTransformer?

Real datasets have **mixed feature types** — numerical columns need scaling, categorical columns need encoding. ColumnTransformer lets you apply **different transformations to different columns** simultaneously, then combine the results into one matrix.

---

### 3. Why should preprocessing happen after train-test split?

If you scale or impute **before** splitting, statistics (mean, std, most-frequent value) are computed on the **full dataset including test data**. This leaks information about the test set into your model. Always: split first → fit preprocessor on train → transform both train and test.

---

### 4. Why is data leakage dangerous?

Leakage gives the model information during training that **won't be available at prediction time**. The model learns a cheat code that doesn't exist in the real world. Validation scores look great; real-world performance is terrible. It's one of the most common ways ML projects fail silently.

---

### 5. fit vs transform vs fit_transform

| Method | What it does | When to use |
|---|---|---|
| **fit** | Learns parameters from data (mean, std, etc.) | Only on training data |
| **transform** | Applies learned parameters to data | On train AND test |
| **fit_transform** | Learns + applies in one step | Convenience for training data only |

> **Never** call `fit` or `fit_transform` on test data.

---

### 6. Why use cross-validation?

A single train/test split can be **lucky or unlucky** — you might get an unusually easy test set. Cross-validation (e.g., k-fold) trains and evaluates k times on different data splits, giving a **more reliable estimate** of true performance with less variance.

---

### 7. Why might validation score differ from test score?

- You **tuned hyperparameters on validation** → validation is no longer truly unseen → optimistic estimate
- Different data distributions across splits (small dataset, uneven class balance)
- The test set is harder/different by chance
- Solution: nested CV or a held-out final test set touched only once

---

### 8. Why can adding more features hurt performance?

- **Curse of dimensionality**: more features = sparser data space, harder to generalize
- More noise features = more ways for the model to overfit
- Irrelevant features add variance without reducing bias
- Linear models especially: they'll assign non-zero weight to junk features (use regularization or feature selection)

---

### 9. Why is feature engineering often more important than model complexity?

Models can only work with the information they're given. A better **representation** of the problem (right features, right transformations) is often worth more than a fancier algorithm on raw features. Example: adding `price_per_sqft` to a housing model can be more powerful than switching from Ridge to XGBoost.

---

### 10. Why can tree models work without scaling?

Decision trees split on **thresholds**: "is feature X > 5?" — the actual scale doesn't matter, only the relative ordering of values. Whether income is in dollars or thousands-of-dollars, the split point just changes proportionally. Scaling doesn't affect tree structure.

---

### 11. Why do linear models care more about scaling?

Linear models compute **weighted sums**: prediction = β₁x₁ + β₂x₂ + ... The gradient descent optimization and regularization both depend on the **magnitude** of coefficients. Unscaled features lead to unequal gradient steps and unfair regularization.

---

### 12. What is overfitting?

The model learns the **training data too well** — including its noise and random fluctuations — instead of the underlying pattern. It performs great on training data but poorly on new data. Fix: regularization, more data, simpler model, dropout, early stopping.

---

### 13. What is underfitting?

The model is **too simple** to capture the true pattern in the data. It performs poorly on both training and test data. Fix: more complex model, more features, less regularization, more training.

---

### 14. Bias vs Variance

| | Bias | Variance |
|---|---|---|
| **Definition** | Error from wrong assumptions (model too simple) | Error from sensitivity to training data fluctuations |
| **Symptom** | Underfitting — bad on train AND test | Overfitting — great on train, bad on test |
| **Cause** | Model can't capture complexity | Model memorizes noise |
| **Fix** | More complexity, better features | Regularization, more data, simpler model |

> **Bias-variance tradeoff**: decreasing one tends to increase the other. The goal is the sweet spot.

---

### 15. Why can simpler models sometimes generalize better?

With limited data, a complex model has many parameters to fit — it can always find a way to memorize the training set. A simple model has fewer degrees of freedom; it's forced to learn only the strongest, most consistent patterns. On small/noisy datasets, this **lower variance wins** over the complex model's ability to fit anything.

---

### 16. Why are ensembles powerful?

A single model has one "view" of the data. Ensembles combine **many diverse models**:
- **Bagging** (Random Forest): reduces variance by averaging uncorrelated models
- **Boosting** (XGBoost): reduces bias by sequentially correcting errors
- Errors of individual models tend to **cancel out** in aggregate

---

### 17. When to choose linear model vs tree model?

| Situation | Choose |
|---|---|
| Small dataset | **Linear** |
| Need interpretability / auditability | **Linear** |
| Linear or near-linear relationships | **Linear** |
| Large dataset with complex non-linear patterns | **Tree** |
| Mixed feature types, missing data, outliers | **Tree** |
| Need raw predictive power | **Tree** (especially gradient boosting) |
| Fast training and inference | **Linear** |

---

### 18. Why are metrics chosen differently for regression vs classification?

They measure fundamentally different things:
- **Regression** predicts a continuous number → metric must measure distance/magnitude of error (MAE, RMSE, R²)
- **Classification** predicts a category → metric measures correctness, with nuances like class imbalance (accuracy, F1, AUC-ROC, precision, recall)

Using regression metrics on classification or vice versa would be meaningless.

---

### 19. Housing Project Workflow — End to End

```
1. DEFINE THE PROBLEM
   └── Target: predict house price (regression)
   └── Metric: RMSE (penalizes large errors; errors in price are costly)

2. EXPLORE THE DATA (EDA)
   └── Distribution of target (skewed? log-transform?)
   └── Missing values, outliers, feature types
   └── Correlations: heatmap, scatter plots vs target

3. TRAIN-TEST SPLIT (before any preprocessing)
   └── Typical: 80/20 split, stratified if needed
   └── Test set is locked — not touched until final evaluation

4. PREPROCESSING (fit only on train)
   ├── Numerical: impute missing → scale (StandardScaler)
   ├── Categorical: impute missing → encode (OneHotEncoder)
   └── ColumnTransformer combines both

5. PIPELINE
   └── Pipeline([('preprocessor', ct), ('model', Ridge())])
   └── One object handles all steps cleanly

6. BASELINE MODEL
   └── LinearRegression → establishes lower bound
   └── Check train vs test R²: is there overfitting?

7. REGULARIZED MODELS
   └── Ridge, Lasso, ElasticNet
   └── Cross-validate to tune alpha (GridSearchCV or RandomizedSearchCV)

8. EVALUATE
   └── RMSE, MAE, R² on test set (once only)
   └── Plot residuals: are they random? (validates assumptions)
   └── Plot predicted vs actual

9. INTERPRET
   └── Extract coefficients from best model
   └── Identify top features driving price

10. ITERATE
    └── Feature engineering (price_per_room, log_sqft, etc.)
    └── Try non-linear models if linear is insufficient
    └── Report final test performance
```

---

*Built for retention — each answer complete enough to need no further lookup.*

| Model                        | When to Use                                                         | Avoid When                                                    |
| ---------------------------- | ------------------------------------------------------------------- | ------------------------------------------------------------- |
| `Linear Regression`          | Simple linear-ish relationships, clean data, interpretable baseline | Strong multicollinearity, overfitting, highly nonlinear data  |
| `Multiple Linear Regression` | Multiple useful features affect target together                     | Features heavily correlated or nonlinear patterns dominate    |
| `Ridge Regression`           | Multicollinearity, many correlated features, overfitting risk       | Need automatic feature selection/sparse model                 |
| `Lasso Regression`           | Feature selection needed, many weak/useless features                | Highly correlated important features                          |
| `ElasticNet`                 | Many correlated features + want some feature selection              | Small simple datasets where Ridge/Lasso alone works fine      |
| `Polynomial Regression`      | Curved/nonlinear relationships but still structured                 | High-dimensional noisy data, large degree causing overfitting |


| Model Type        | Accuracy Potential        | Training Speed | Interpretability |
| ----------------- | ------------------------- | -------------- | ---------------- |
| Linear Regression | Lower                     | Very fast      | Very high        |
| Ridge/Lasso       | Slightly better           | Very fast      | High             |
| Decision Tree     | Better nonlinear handling | Fast           | Medium           |
| Random Forest     | Strong                    | Slower         | Lower            |
| XGBoost           | Very strong               | Slower/heavier | Lower            |
| Deep Learning     | Can be extremely strong   | Heavy          | Very low         |


| Model                 | Example Use Case                                                              |
| --------------------- | ----------------------------------------------------------------------------- |
| `Linear Regression`   | Predicting house prices from area, rooms, location                            |
| `Ridge Regression`    | Salary prediction with many correlated employee/company features              |
| `Lasso Regression`    | Gene-selection problem where thousands of features exist but only some matter |
| `ElasticNet`          | Real-estate or financial datasets with many correlated useful features        |
| `Decision Tree`       | Loan approval rules that depend on conditions/thresholds                      |
| `Random Forest`       | Customer churn prediction on messy business data                              |
| `XGBoost`             | Kaggle competitions, fraud detection, ranking systems                         |
| `Logistic Regression` | Spam detection, pass/fail prediction, medical yes/no classification           |
| `SVM`                 | Medium-sized classification with clear class separation                       |
| `KNN`                 | Small recommendation/s                                                        |
